### Quality Control
This notebook examines the output netCDF files, and does some basic quality control checks to make sure values in the netCDFs match their sources in the CSV files.

In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import os
import random
from pathlib import Path
from luts import *

# source file directory
stats_dir = Path('/beegfs/CMIP6/jdpaul3/hydroviz_data/stats')
# output directory
nc_dir = Path('/beegfs/CMIP6/jdpaul3/hydroviz_data/maurer/nc_stats_fix')

# get stats list
stats = list(stat_vars_dict.keys())

In [2]:
# first read stats CSVs and do some filtering of results ...
# this is similar to the code in filter_files() in functions.py, but runs independently here
files = stats_dir.glob('*.csv')
seg_files = []
hru_files = []
for f in files:
    if "diff" not in f.name: pass
    elif "1952_2005" in f.name: pass
    elif "Maurer" in f.name: pass
    elif "hru" in f.name:
        hru_files.append(f)
    else: 
        seg_files.append(f)

In [3]:
# load netCDFs
seg = xr.open_dataset(os.path.join(nc_dir, "seg_diff.nc"))
hru = xr.open_dataset(os.path.join(nc_dir, "hru_diff.nc"))

In [4]:
# check out the datasets - structure should be identical except for length of stream_ids
seg

<xarray.Dataset> Size: 4GB
Dimensions:    (landcover: 2, model: 13, scenario: 4, era: 3, stream_id: 56460)
Coordinates:
  * landcover  (landcover) float32 8B 0.0 1.0
  * model      (model) float32 52B 0.0 1.0 2.0 3.0 4.0 ... 10.0 11.0 12.0 13.0
  * scenario   (scenario) float32 16B 1.0 2.0 3.0 4.0
  * era        (era) float32 12B 1.0 2.0 3.0
  * stream_id  (stream_id) int32 226kB 1 2 3 4 5 ... 56457 56458 56459 56460
Data variables: (12/52)
    dh1        (landcover, model, scenario, era, stream_id) float32 70MB ...
    dh2        (landcover, model, scenario, era, stream_id) float32 70MB ...
    dh3        (landcover, model, scenario, era, stream_id) float32 70MB ...
    dh4        (landcover, model, scenario, era, stream_id) float32 70MB ...
    dh5        (landcover, model, scenario, era, stream_id) float32 70MB ...
    dh15       (landcover, model, scenario, era, stream_id) float32 70MB ...
    ...         ...
    ra3        (landcover, model, scenario, era, stream_id) float32 70MB ...
    ra8        (landcover, model, scenario, era, stream_id) float32 70MB ...
    spr_ord    (landcover, model, scenario, era, stream_id) float32 70MB ...
    sum_ord    (landcover, model, scenario, era, stream_id) float32 70MB ...
    th1        (landcover, model, scenario, era, stream_id) float32 70MB ...
    tl1        (landcover, model, scenario, era, stream_id) float32 70MB ...
Attributes:
    Data Source:         {'Title': 'Model Input and Output for Hydrologic Sim...
    CMIP5 GCM Metadata:  {'ACCESS1-0': {'Modeling Center': 'Commonwealth Scie...

In [5]:
hru

<xarray.Dataset> Size: 7GB
Dimensions:    (landcover: 2, model: 13, scenario: 4, era: 3, stream_id: 109951)
Coordinates:
  * landcover  (landcover) float32 8B 0.0 1.0
  * model      (model) float32 52B 0.0 1.0 2.0 3.0 4.0 ... 10.0 11.0 12.0 13.0
  * scenario   (scenario) float32 16B 1.0 2.0 3.0 4.0
  * era        (era) float32 12B 1.0 2.0 3.0
  * stream_id  (stream_id) int64 880kB 2118 1865 1874 ... 109450 109495 109568
Data variables: (12/52)
    dh1        (landcover, model, scenario, era, stream_id) float32 137MB ...
    dh2        (landcover, model, scenario, era, stream_id) float32 137MB ...
    dh3        (landcover, model, scenario, era, stream_id) float32 137MB ...
    dh4        (landcover, model, scenario, era, stream_id) float32 137MB ...
    dh5        (landcover, model, scenario, era, stream_id) float32 137MB ...
    dh15       (landcover, model, scenario, era, stream_id) float32 137MB ...
    ...         ...
    ra3        (landcover, model, scenario, era, stream_id) float32 137MB ...
    ra8        (landcover, model, scenario, era, stream_id) float32 137MB ...
    spr_ord    (landcover, model, scenario, era, stream_id) float32 137MB ...
    sum_ord    (landcover, model, scenario, era, stream_id) float32 137MB ...
    th1        (landcover, model, scenario, era, stream_id) float32 137MB ...
    tl1        (landcover, model, scenario, era, stream_id) float32 137MB ...
Attributes:
    Data Source:         {'Title': 'Model Input and Output for Hydrologic Sim...
    CMIP5 GCM Metadata:  {'ACCESS1-0': {'Modeling Center': 'Commonwealth Scie...

In [6]:
# create a testing function that parses the file name and uses the coords to check the contents of the netCDF
# contents for every statistic are compared to data in the CSV
# this test is only run for stream segments, not HRUs

def test_nc(ds, test_csvs):
    id_col = 'seg_id'
    nc_ids = ds['stream_id'].values
    
    for csv in test_csvs:
        # Read CSV with ID column plus all stats
        df = pd.read_csv(csv, usecols=[id_col] + stats)
        df.replace(-99999, np.nan, inplace=True)
        
        # Filter CSV to only include IDs present in the netCDF
        df = df[df[id_col].isin(nc_ids)].set_index(id_col).reindex(nc_ids).reset_index()

        parts = csv.name.split('_')
        try:
            landcover, model, scenario, era = parts[0], parts[1], parts[2], "_".join([parts[5], parts[6].split(".")[0]])
        except:
            print(f"Error parsing file: {csv.name}")
            continue

        # parse each part of the filename into integers using the encodings_lookup dict
        landcover_int = encodings_lookup['landcover'].get(landcover, landcover)
        model_int = encodings_lookup['model'].get(model, model)
        scenario_int = encodings_lookup['scenario'].get(scenario, scenario)
        era_int = encodings_lookup['era'].get(era, era)

        for stat in stats:
            sel_dict = {"landcover": landcover_int, "model": model_int, "scenario": scenario_int, "era": era_int}
            values = ds[stat].sel(sel_dict).load().values
            csv_values = df[stat].values
            
            assert np.allclose(values, csv_values, equal_nan=True), f"Error in dataset: values for {stat} do not match value in {csv.name}"
            break


In [7]:
# for each geometry, pick 30 random files (~10%) and run the tests
# a successful test will produce no output

seg_test_files = random.sample(seg_files, 30)
test_nc(seg, seg_test_files)